# StockFlow AI — Smart Restock Predictor & Discount Engine

**Kategori:** Smart Commerce & Smart Logistics
**Target pengguna:** UMKM / retail kecil-menengah

StockFlow AI membantu toko menjawab dua pertanyaan operasional sehari-hari:

- **Fitur A — Smart Restock Predictor:** dari histori penjualan 30 hari terakhir, berapa unit yang
  harus di-restock untuk memenuhi permintaan 7 hari ke depan tanpa kehabisan stok (dan tanpa
  restock berlebihan)?
- **Fitur B — Discount Engine:** untuk barang yang menumpuk (overstock / mendekati kedaluwarsa),
  diskon berapa persen yang **kemungkinan besar** menghabiskan stok dalam target hari, tapi harga
  jual tetap di atas harga modal (`cogs`) sehingga toko tidak rugi?

Notebook ini mengikuti alur end-to-end: **EDA → data handling → feature engineering →
preprocessing → modelling → hyperparameter tuning → evaluasi & pemilihan model terbaik →
deployment**.

## Dataset

File: `stockflow_dataset.csv` — 5.475 baris (15 SKU × 365 hari), data sintetik yang ditanam
mengikuti proses `demand ~ Poisson(λ)` dengan faktor hari, gajian, Ramadan, tren, dan elastisitas
harga per SKU (lihat `STOCKFLOW_SINGLEFILE_README.md` untuk kamus data lengkap).

Tiga catatan metodologi penting dari dokumentasi dataset yang **diikuti secara eksplisit** di
notebook ini:

1. **Sensor stok (censoring).** Saat stok habis, `quantity_sold` terpotong — permintaan asli ada
   di `lost_sales`. Untuk estimasi permintaan (demand), kita pakai
   `true_demand = quantity_sold + lost_sales`, bukan `quantity_sold` saja.
2. **Endogenitas harga.** Diskon sering dipicu oleh stok menumpuk / mendekati kedaluwarsa, bukan
   acak. Model demand harus mengontrol `hari`, `is_periode_gajian`, `is_ramadan`, dan tren waktu
   agar efek harga tidak bias.
3. **Split temporal, bukan acak.** Data dibagi berdasarkan tanggal (train: 18 Agu 2025 – 17 Jul
   2026, test: 18 Jul – 17 Agu 2026) supaya evaluasi mencerminkan kondisi nyata (forecast ke masa
   depan, bukan interpolasi).

## 1. Setup & Imports

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, PoissonRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_poisson_deviance

import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import joblib
from pathlib import Path

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (11, 4.5)
plt.rcParams["axes.titleweight"] = "bold"

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

print("Environment ready.")

Environment ready.


## 2. Load Data

In [2]:
DATA_PATH = "stockflow_dataset.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["tanggal", "tanggal_kedaluwarsa"])
df = df.sort_values(["product_id", "tanggal"]).reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Periode: {df['tanggal'].min().date()} -> {df['tanggal'].max().date()}")
print(f"Jumlah SKU: {df['product_id'].nunique()}")
df.head()

Shape: (5475, 23)
Periode: 2025-08-18 -> 2026-08-17
Jumlah SKU: 15


,tanggal,hari,product_id,product_name,kategori,satuan,harga_normal,diskon_persen,harga_jual,cogs,quantity_sold,lost_sales,revenue,gross_profit,stock_awal,restock_diterima,stock_akhir,restock_dipesan,expired_qty,tanggal_kedaluwarsa,days_to_expiry,is_periode_gajian,is_ramadan
0,2025-08-18,Senin,SKU001,Minyak Goreng Kemasan 2L,Sembako,botol,34000,0.0,34000,28500,11,0,374000,60500,121,0,110,0,0,2026-08-29,376,0,0
1,2025-08-19,Selasa,SKU001,Minyak Goreng Kemasan 2L,Sembako,botol,34000,0.0,34000,28500,18,0,612000,99000,110,0,92,0,0,2026-08-29,375,0,0
2,2025-08-20,Rabu,SKU001,Minyak Goreng Kemasan 2L,Sembako,botol,34000,0.0,34000,28500,15,0,510000,82500,92,0,77,0,0,2026-08-29,374,0,0
3,2025-08-21,Kamis,SKU001,Minyak Goreng Kemasan 2L,Sembako,botol,34000,0.0,34000,28500,20,0,680000,110000,77,0,57,0,0,2026-08-29,373,0,0
4,2025-08-22,Jumat,SKU001,Minyak Goreng Kemasan 2L,Sembako,botol,34000,0.0,34000,28500,22,0,748000,121000,57,0,35,150,0,2026-08-29,372,0,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5475 entries, 0 to 5474
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   tanggal              5475 non-null   datetime64[ns]
 1   hari                 5475 non-null   object        
 2   product_id           5475 non-null   object        
 3   product_name         5475 non-null   object        
 4   kategori             5475 non-null   object        
 5   satuan               5475 non-null   object        
 6   harga_normal         5475 non-null   int64         
 7   diskon_persen        5475 non-null   float64       
 8   harga_jual           5475 non-null   int64         
 9   cogs                 5475 non-null   int64         
 10  quantity_sold        5475 non-null   int64         
 11  lost_sales           5475 non-null   int64         
 12  revenue              5475 non-null   int64         
 13  gross_profit         5475 non-nul

## 3. Exploratory Data Analysis (EDA)

In [4]:
print("Missing values per kolom:")
print(df.isna().sum()[df.isna().sum() > 0])

print(f"\nDuplikat (product_id, tanggal): {df.duplicated(['product_id', 'tanggal']).sum()}")

full_range = pd.date_range(df["tanggal"].min(), df["tanggal"].max(), freq="D")
coverage = df.groupby("product_id")["tanggal"].nunique()
print(f"\nSetiap SKU harus punya {len(full_range)} baris (1 tahun penuh):")
print(coverage[coverage != len(full_range)] if (coverage != len(full_range)).any() else "OK - semua SKU lengkap.")

print(f"\nBaris harga_jual < cogs (margin negatif): {(df['harga_jual'] < df['cogs']).sum()}")
print(f"Baris gross_profit < 0: {(df['gross_profit'] < 0).sum()}")

Missing values per kolom:
Series([], dtype: int64)

Duplikat (product_id, tanggal): 0

Setiap SKU harus punya 365 baris (1 tahun penuh):
OK - semua SKU lengkap.

Baris harga_jual < cogs (margin negatif): 0
Baris gross_profit < 0: 0


Data lengkap (365 hari x 15 SKU, tanpa duplikat/hilang) dan **tidak ada satu baris pun** dengan
harga jual di bawah modal — konsisten dengan klaim di README bahwa `cogs` adalah batas bawah
keras yang sudah dijaga saat data dibuat. Ini juga jadi acuan: model Fitur B **wajib** menjaga
batas ini sendiri untuk data nyata yang mungkin tidak serapi ini.

In [5]:
daily_total = df.groupby("tanggal")["quantity_sold"].sum()

fig, ax = plt.subplots(figsize=(13, 4.5))
daily_total.rolling(7).mean().plot(ax=ax, label="Rata-rata bergerak 7 hari", linewidth=2)
daily_total.plot(ax=ax, alpha=0.25, label="Harian (raw)")
ax.set_title("Total Unit Terjual per Hari (Semua SKU)")
ax.set_xlabel("")
ax.set_ylabel("Unit terjual")
ax.legend()
plt.tight_layout()
plt.show()

In [6]:
cat_daily = df.groupby(["tanggal", "kategori"])["quantity_sold"].sum().unstack()

fig, ax = plt.subplots(figsize=(13, 5))
cat_daily.rolling(7).mean().plot(ax=ax, linewidth=1.6)
ax.set_title("Tren Penjualan per Kategori (rata-rata bergerak 7 hari)")
ax.set_ylabel("Unit terjual")
ax.set_xlabel("")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

In [7]:
dow_order = ["Senin", "Selasa", "Rabu", "Kamis", "Jumat", "Sabtu", "Minggu"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))

sns.boxplot(data=df, x="hari", y="quantity_sold", order=dow_order, ax=axes[0], showfliers=False)
axes[0].set_title("Efek Hari-dalam-Minggu")
axes[0].tick_params(axis="x", rotation=40)

sns.boxplot(data=df, x="is_periode_gajian", y="quantity_sold", ax=axes[1], showfliers=False)
axes[1].set_title("Efek Periode Gajian (tgl 25-5)")
axes[1].set_xticklabels(["Bukan gajian", "Gajian"])

sns.boxplot(data=df, x="is_ramadan", y="quantity_sold", ax=axes[2], showfliers=False)
axes[2].set_title("Efek Ramadan")
axes[2].set_xticklabels(["Bukan Ramadan", "Ramadan"])

plt.tight_layout()
plt.show()

print(df.groupby("hari")["quantity_sold"].mean().reindex(dow_order).round(2))

hari
Senin     14.64
Selasa    15.74
Rabu      16.86
Kamis     17.57
Jumat     20.31
Sabtu     21.64
Minggu    19.84
Name: quantity_sold, dtype: float64


In [8]:
# Elastisitas harga: hubungan log(price_ratio) vs log(demand) untuk beberapa SKU dengan
# variasi diskon terbesar. price_ratio = harga_jual / harga_normal (BUKAN harga_normal saja,
# sesuai catatan README - itu yang benar-benar menggerakkan permintaan).
df["price_ratio"] = df["harga_jual"] / df["harga_normal"]
df["true_demand"] = df["quantity_sold"] + df["lost_sales"]  # koreksi sensor stok-habis

sample_skus = ["SKU001", "SKU006", "SKU012", "SKU013"]

fig, axes = plt.subplots(1, 4, figsize=(16, 3.6), sharey=False)
for ax, sku in zip(axes, sample_skus):
    d = df[(df["product_id"] == sku) & (df["true_demand"] > 0)]
    name = d["product_name"].iloc[0]
    ax.scatter(np.log(d["price_ratio"]), np.log(d["true_demand"]), s=8, alpha=0.35)
    ax.set_title(f"{sku}\n{name}", fontsize=9)
    ax.set_xlabel("log(harga_jual / harga_normal)")
axes[0].set_ylabel("log(true_demand)")
plt.tight_layout()
plt.show()

Semakin curam kemiringan (lebih negatif), semakin elastis produk itu terhadap diskon — terlihat
jelas SKU012 (Roti Tawar) jauh lebih curam dibanding SKU001 (Minyak Goreng), sesuai tabel
elastisitas di README (-3.1 vs -2.1).

In [9]:
stockout_rate = df["is_stockout"] = (df["stock_akhir"] == 0)
print(f"Persentase hari stok habis: {df['is_stockout'].mean() * 100:.1f}%")
print(f"Lost sales sebagai % dari true demand: {df['lost_sales'].sum() / df['true_demand'].sum() * 100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

so_by_sku = df.groupby("product_id")["is_stockout"].mean().sort_values(ascending=False) * 100
so_by_sku.plot(kind="bar", ax=axes[0], color="indianred")
axes[0].set_title("Persentase Hari Stok Habis per SKU")
axes[0].set_ylabel("%")

promo_by_sku = (df.groupby("product_id")["diskon_persen"].apply(lambda s: (s > 0).mean()) * 100).sort_values(ascending=False)
promo_by_sku.plot(kind="bar", ax=axes[1], color="steelblue")
axes[1].set_title("Persentase Hari Ada Promo per SKU")
axes[1].set_ylabel("%")

plt.tight_layout()
plt.show()

Persentase hari stok habis: 5.9%
Lost sales sebagai % dari true demand: 4.0%


**Ini adalah bukti langsung sensor stok:** hari-hari `stock_akhir == 0` memotong `quantity_sold`
di bawah permintaan sebenarnya. Kalau model demand dilatih dari `quantity_sold` mentah pada SKU
dengan stockout rate tinggi, model akan sistematis **meremehkan** permintaan asli — makanya
Fitur A memakai `true_demand = quantity_sold + lost_sales` sebagai target.

In [10]:
numeric_cols_corr = ["price_ratio", "diskon_persen", "true_demand", "quantity_sold", "lost_sales",
                      "stock_akhir", "days_to_expiry", "is_periode_gajian", "is_ramadan"]

corr = df[numeric_cols_corr].corr()

fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Korelasi Antar Fitur Numerik")
plt.tight_layout()
plt.show()

### Ringkasan insight EDA

- Ada pola mingguan yang kuat (Jumat-Minggu lebih tinggi) dan lonjakan jelas saat periode gajian
  & Ramadan — ini semua harus jadi fitur kalender di model, bukan diabaikan.
- Elastisitas harga bervariasi tajam antar SKU (produk segar/cepat basi jauh lebih elastis) - model
  pooled awalnya mencoba membedakan tiap SKU lewat fitur `product_id`, tapi ini dijatuhkan (lihat
  Bagian 5.1/5.2): aplikasi produksi membuat produk baru secara dinamis, jadi identitas SKU mentah
  tidak pernah bisa digeneralisasi ke produk yang belum pernah dilihat model. Fitur `kategori` saja
  yang dipakai sebagai sinyal kategorikal.
- Stockout terjadi cukup sering di sebagian SKU sehingga `quantity_sold` mentah bias ke bawah —
  wajib pakai `true_demand`.
- `diskon_persen` dan `price_ratio` berkorelasi kuat (redundan secara matematis) — dipakai salah
  satu saja di model demand-response supaya tidak multikolinearitas.

## 4. Data Handling

In [11]:
# --- 4.1 Bersihkan & turunkan kolom kunci -----------------------------------
df["is_stockout"] = (df["stock_akhir"] == 0).astype(int)
df["trend"] = (df["tanggal"] - df["tanggal"].min()).dt.days  # indeks tren waktu global
df["dow_num"] = df["tanggal"].dt.dayofweek  # 0=Senin ... 6=Minggu
df["dow_sin"] = np.sin(2 * np.pi * df["dow_num"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["dow_num"] / 7)
df["kategori"] = df["kategori"].astype("category")
df["product_id"] = df["product_id"].astype("category")

assert df["true_demand"].ge(0).all(), "Ditemukan demand negatif - cek data."
assert (df["harga_jual"] >= df["cogs"]).all(), "Ditemukan harga_jual < cogs."

print("Kolom turunan siap:", ["price_ratio", "true_demand", "is_stockout", "trend", "dow_sin", "dow_cos"])
df[["tanggal", "product_id", "quantity_sold", "lost_sales", "true_demand", "is_stockout"]].head()

Kolom turunan siap: ['price_ratio', 'true_demand', 'is_stockout', 'trend', 'dow_sin', 'dow_cos']


,tanggal,product_id,quantity_sold,lost_sales,true_demand,is_stockout
0,2025-08-18,SKU001,11,0,11,0
1,2025-08-19,SKU001,18,0,18,0
2,2025-08-20,SKU001,15,0,15,0
3,2025-08-21,SKU001,20,0,20,0
4,2025-08-22,SKU001,22,0,22,0


In [12]:
# --- 4.2 Split temporal (bukan acak) -----------------------------------------
# Train: 18 Agu 2025 - 17 Jul 2026 | Test: 18 Jul - 17 Agu 2026 (30 hari terakhir, out-of-time)
TEST_START = pd.Timestamp("2026-07-18")

print(f"Train : {df['tanggal'].min().date()} -> {(TEST_START - pd.Timedelta(days=1)).date()}"
      f"  ({(df['tanggal'] < TEST_START).sum()} baris)")
print(f"Test  : {TEST_START.date()} -> {df['tanggal'].max().date()}"
      f"  ({(df['tanggal'] >= TEST_START).sum()} baris)")

Train : 2025-08-18 -> 2026-07-17  (5010 baris)
Test  : 2026-07-18 -> 2026-08-17  (465 baris)


Split ini **temporal**, bukan `train_test_split` acak. Kalau kita split acak, hari-hari di masa
depan bisa masuk ke train dan "membocorkan" informasi tren/musiman ke model sebelum dievaluasi di
data yang lebih tua — skor akan terlihat bagus di notebook tapi runtuh saat dipakai untuk
forecast sungguhan. Prinsip yang sama dipakai untuk `TimeSeriesSplit` saat cross-validation di
bagian hyperparameter tuning.

## 5. Feature Engineering

### 5.1 Fitur A — Smart Restock Predictor

Target: **`target_7d`** = total `true_demand` selama 7 hari ke depan (t+1 .. t+7), diprediksi dari
fitur yang hanya memakai informasi sampai akhir hari t (*direct multi-step forecasting* — lebih
stabil untuk MVP dibanding forecasting rekursif 1-hari yang mengakumulasi error).

Fitur yang dipakai:
- **Lag & rolling window** dari `true_demand` (bukan `quantity_sold`) — menangkap level & tren
  permintaan terbaru.
- **Kalender**: hari dalam minggu (encoding sin/cos), tren waktu, serta *fraksi* hari gajian/
  Ramadan dalam 7 hari ke depan (info kalender selalu diketahui di muka, jadi ini bukan
  kebocoran data).
- **Stok saat ini** (`stock_akhir` hari t) — dipakai nanti untuk menghitung kuantitas restock,
  bukan cuma prediksi demand-nya saja.
- **Kategori** sebagai fitur kategorikal (pooled model lintas SKU, bukan satu model per SKU). Fitur
  identitas `product_id` mentah sengaja TIDAK dipakai: aplikasi produksi membuat produk baru secara
  dinamis, jadi SKU mentah tidak pernah bisa digeneralisasi ke produk yang belum pernah dilihat
  model saat training - lebih baik pooled murni lewat `kategori`.

In [13]:
def future_sum(s: pd.Series, horizon: int = 7) -> pd.Series:
    # Jumlah horizon hari ke depan (t+1..t+horizon), tanpa memakai nilai hari t sendiri.
    reversed_roll = s[::-1].rolling(window=horizon, min_periods=horizon).sum()[::-1]
    return reversed_roll.shift(-1)


def calc_is_gajian(date):
    d = pd.Timestamp(date).day
    return int(d >= 25 or d <= 5)


def calc_is_ramadan(date):
    d = pd.Timestamp(date)
    return int(pd.Timestamp("2026-02-17") <= d <= pd.Timestamp("2026-03-19"))


def frac_next7(date, rule_fn, horizon=7):
    # Dihitung dari ATURAN kalender (bukan lookup baris masa depan di dataframe), supaya tetap
    # bisa dihitung untuk tanggal terakhir di dataset - persis saat prediksi restock dibutuhkan.
    future_dates = [pd.Timestamp(date) + pd.Timedelta(days=i) for i in range(1, horizon + 1)]
    return float(np.mean([rule_fn(d) for d in future_dates]))


feat = df.copy()
g = feat.groupby("product_id", observed=True)["true_demand"]

# --- lag & rolling (semua dihitung dari histori s.d. hari t -> aman dari leakage) ---
for lag in [1, 7, 14, 28]:
    feat[f"lag_{lag}"] = g.shift(lag)
feat["demand_today"] = feat["true_demand"]
feat["roll_mean_7"] = g.transform(lambda s: s.rolling(7, min_periods=3).mean())
feat["roll_std_7"] = g.transform(lambda s: s.rolling(7, min_periods=3).std())
feat["roll_mean_14"] = g.transform(lambda s: s.rolling(14, min_periods=5).mean())
feat["roll_mean_28"] = g.transform(lambda s: s.rolling(28, min_periods=7).mean())

# --- target: total demand 7 hari ke depan ---
feat["target_7d"] = feat.groupby("product_id", observed=True)["true_demand"].transform(future_sum)

# --- fraksi hari gajian/ramadan dalam 7 hari ke depan (info kalender, bukan kebocoran) ---
unique_dates = feat["tanggal"].drop_duplicates()
gajian_frac_map = {d: frac_next7(d, calc_is_gajian) for d in unique_dates}
ramadan_frac_map = {d: frac_next7(d, calc_is_ramadan) for d in unique_dates}
feat["is_periode_gajian_frac_7d"] = feat["tanggal"].map(gajian_frac_map)
feat["is_ramadan_frac_7d"] = feat["tanggal"].map(ramadan_frac_map)

FEATURES_A_NUM = [
    "lag_1", "lag_7", "lag_14", "lag_28", "demand_today",
    "roll_mean_7", "roll_std_7", "roll_mean_14", "roll_mean_28",
    "stock_akhir", "price_ratio", "trend", "dow_sin", "dow_cos",
    "is_periode_gajian", "is_ramadan", "is_periode_gajian_frac_7d", "is_ramadan_frac_7d",
]
FEATURES_A_CAT = ["kategori"]  # product_id dropped: live app creates new products dynamically,
# so a raw product-id identity feature would always be "unseen" at inference time (see README).
TARGET_A = "target_7d"

model_a_df = feat.dropna(subset=FEATURES_A_NUM + [TARGET_A]).reset_index(drop=True)
print(f"Baris tersisa setelah drop NaN (butuh histori 28 hari & target 7 hari): {len(model_a_df)} / {len(feat)}")

# Frame terpisah untuk *inference* (hanya butuh histori 28 hari, TIDAK butuh target_7d -
# baris 7 hari terakhir per SKU tidak punya target_7d karena masa depannya belum terjadi,
# tapi baris itu justru yang paling penting untuk prediksi restock "hari ini").
feat_inference_a = feat.dropna(subset=FEATURES_A_NUM).reset_index(drop=True)

model_a_df[["product_id", "tanggal"] + FEATURES_A_NUM + [TARGET_A]].head()

Baris tersisa setelah drop NaN (butuh histori 28 hari & target 7 hari): 4950 / 5475


,product_id,tanggal,lag_1,lag_7,lag_14,lag_28,demand_today,roll_mean_7,roll_std_7,roll_mean_14,roll_mean_28,stock_akhir,price_ratio,trend,dow_sin,dow_cos,is_periode_gajian,is_ramadan,is_periode_gajian_frac_7d,is_ramadan_frac_7d,target_7d
0,SKU001,2025-09-15,26.0,25.0,20.0,11.0,16,20.571429,4.429339,22.428571,22.785714,73,1.000000,28,0.000000,1.000000,0,0,0.000000,0.0,136.0
1,SKU001,2025-09-16,16.0,27.0,25.0,18.0,25,20.285714,3.988077,22.428571,23.035714,48,0.900000,29,0.781831,0.623490,0,0,0.000000,0.0,132.0
2,SKU001,2025-09-17,25.0,21.0,23.0,15.0,20,20.142857,3.976119,22.214286,23.214286,28,0.850000,30,0.974928,-0.222521,0,0,0.000000,0.0,137.0
3,SKU001,2025-09-18,20.0,19.0,30.0,20.0,16,19.714286,4.270608,21.214286,23.071429,162,0.879412,31,0.433884,-0.900969,0,0,0.142857,0.0,144.0
4,SKU001,2025-09-19,16.0,16.0,21.0,22.0,22,20.571429,3.994043,21.285714,23.071429,140,0.850000,32,-0.433884,-0.900969,0,0,0.285714,0.0,143.0


### 5.2 Fitur B — Discount Engine

Target: **`true_demand`** harian, dimodelkan sebagai proses hitung (count) — cocok dengan
proses generatif aslinya (`demand ~ Poisson(lambda)`, lihat README). Model ini nanti dipakai
sebagai **mesin simulasi "what-if"**: berapa ekspektasi permintaan harian kalau produk X didiskon
sekian persen, di kondisi kalender tertentu?

Fitur kunci: `log(price_ratio)` (elastisitas harga, **bukan** `harga_normal` atau `diskon_persen`
mentah — keduanya redundan/berkolinear dengan `price_ratio`), dikontrol dengan hari, gajian,
Ramadan, tren, kategori, SKU, `days_to_expiry`, dan `stock_akhir` (mengontrol endogenitas: diskon
sering dipicu stok tinggi / mendekati kedaluwarsa, bukan acak — lihat catatan README).

In [14]:
feat_b = df.copy()
feat_b["log_price_ratio"] = np.log(feat_b["price_ratio"])

FEATURES_B_NUM = [
    "log_price_ratio", "trend", "dow_sin", "dow_cos",
    "is_periode_gajian", "is_ramadan", "days_to_expiry", "stock_akhir",
]
FEATURES_B_CAT = ["kategori"]  # product_id dropped, same reasoning as FEATURES_A_CAT above.
TARGET_B = "true_demand"

model_b_df = feat_b.dropna(subset=FEATURES_B_NUM + [TARGET_B]).reset_index(drop=True)
print(f"Baris untuk model demand-response: {len(model_b_df)}")
model_b_df[["product_id", "tanggal", "diskon_persen"] + FEATURES_B_NUM + [TARGET_B]].head()

Baris untuk model demand-response: 5475


,product_id,tanggal,diskon_persen,log_price_ratio,trend,dow_sin,dow_cos,is_periode_gajian,is_ramadan,days_to_expiry,stock_akhir,true_demand
0,SKU001,2025-08-18,0.0,0.0,0,0.000000,1.000000,0,0,376,110,11
1,SKU001,2025-08-19,0.0,0.0,1,0.781831,0.623490,0,0,375,92,18
2,SKU001,2025-08-20,0.0,0.0,2,0.974928,-0.222521,0,0,374,77,15
3,SKU001,2025-08-21,0.0,0.0,3,0.433884,-0.900969,0,0,373,57,20
4,SKU001,2025-08-22,0.0,0.0,4,-0.433884,-0.900969,0,0,372,35,22


## 6. Data Preprocessing

Dua keluarga model dipakai di tiap fitur, jadi dua gaya preprocessing:

- **Model linear (Linear/Ridge/Poisson Regression):** butuh scaling untuk fitur numerik dan
  one-hot encoding untuk fitur kategorikal (`kategori`, `product_id`).
- **Model berbasis pohon (Random Forest, HistGradientBoosting, LightGBM):** tidak butuh scaling,
  dan bisa memakai kategori native (`category` dtype) langsung — lebih efisien, tidak meledakkan
  dimensi seperti one-hot pada RF.

`ColumnTransformer` dibungkus jadi fungsi supaya konsisten dipakai ulang di kedua fitur (A & B),
dan **selalu di-fit hanya di data train** untuk mencegah kebocoran statistik (mean/std/kategori
dari data test) ke proses training.

In [15]:
def make_linear_preprocessor(num_cols, cat_cols):
    return ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])


def make_tree_preprocessor(num_cols, cat_cols):
    # Tree-based (RF) tidak perlu scaling, tapi tetap perlu OHE karena RandomForest sklearn
    # belum mendukung kategori native.
    return ColumnTransformer([
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])


def split_time(frame, date_col="tanggal", test_start=TEST_START):
    train = frame[frame[date_col] < test_start].reset_index(drop=True)
    test = frame[frame[date_col] >= test_start].reset_index(drop=True)
    return train, test


def wape(y_true, y_pred):
    # Weighted Absolute Percentage Error - metrik standar untuk forecasting demand ritel.
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return np.abs(y_true - y_pred).sum() / max(np.abs(y_true).sum(), 1e-9)


def eval_regression(y_true, y_pred, label):
    return {
        "model": label,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "WAPE": wape(y_true, y_pred),
    }


print("Helper preprocessing & evaluasi siap.")

Helper preprocessing & evaluasi siap.


## 7. Modelling — Fitur A: Smart Restock Predictor

In [16]:
train_a, test_a = split_time(model_a_df)
X_train_a, y_train_a = train_a[FEATURES_A_NUM + FEATURES_A_CAT], train_a[TARGET_A]
X_test_a, y_test_a = test_a[FEATURES_A_NUM + FEATURES_A_CAT], test_a[TARGET_A]

print(f"Train: {X_train_a.shape}  Test: {X_test_a.shape}")
results_a = []

Train: (4590, 19)  Test: (360, 19)


**Baseline 1 — Naive persistence.** Tebakan paling sederhana: demand 7 hari ke depan = demand 7
hari terakhir (`roll_mean_7 * 7`). Model manapun yang dibangun **wajib** mengalahkan ini, kalau
tidak berarti model tidak memberi nilai tambah dibanding aturan sederhana.

In [17]:
pred_naive = X_test_a["roll_mean_7"] * 7
results_a.append(eval_regression(y_test_a, pred_naive, "Naive (rolling 7d x 7)"))
pd.DataFrame(results_a)

,model,MAE,RMSE,WAPE
0,Naive (rolling 7d x 7),29.527778,44.508676,0.21129


In [18]:
# --- Baseline 2: Ridge Regression (linear, ter-scaling & ter-encoding) ---
ridge_pipe = Pipeline([
    ("prep", make_linear_preprocessor(FEATURES_A_NUM, FEATURES_A_CAT)),
    ("model", Ridge(alpha=1.0, random_state=RANDOM_STATE)),
])
ridge_pipe.fit(X_train_a, y_train_a)
pred_ridge = ridge_pipe.predict(X_test_a)
results_a.append(eval_regression(y_test_a, pred_ridge, "Ridge Regression"))

# --- Random Forest ---
rf_pipe = Pipeline([
    ("prep", make_tree_preprocessor(FEATURES_A_NUM, FEATURES_A_CAT)),
    ("model", RandomForestRegressor(n_estimators=300, max_depth=10, min_samples_leaf=3,
                                     random_state=RANDOM_STATE, n_jobs=-1)),
])
rf_pipe.fit(X_train_a, y_train_a)
pred_rf = rf_pipe.predict(X_test_a)
results_a.append(eval_regression(y_test_a, pred_rf, "Random Forest"))

# --- HistGradientBoostingRegressor (native categorical support, sklearn) ---
X_train_a_hgb = X_train_a.copy()
X_test_a_hgb = X_test_a.copy()
hgb = HistGradientBoostingRegressor(
    categorical_features=FEATURES_A_CAT, random_state=RANDOM_STATE, max_iter=300,
)
hgb.fit(X_train_a_hgb, y_train_a)
pred_hgb = hgb.predict(X_test_a_hgb)
results_a.append(eval_regression(y_test_a, pred_hgb, "HistGradientBoosting"))

# --- LightGBM (native categorical support, gradient boosting) ---
lgb_default = lgb.LGBMRegressor(random_state=RANDOM_STATE, n_estimators=300, verbosity=-1)
lgb_default.fit(X_train_a, y_train_a, categorical_feature=FEATURES_A_CAT)
pred_lgb = lgb_default.predict(X_test_a)
results_a.append(eval_regression(y_test_a, pred_lgb, "LightGBM (default)"))

pd.DataFrame(results_a).sort_values("WAPE")

,model,MAE,RMSE,WAPE
3,HistGradientBoosting,15.782798,22.696495,0.112936
4,LightGBM (default),16.397893,24.363777,0.117337
2,Random Forest,16.689650,25.158028,0.119425
1,Ridge Regression,19.298928,29.222882,0.138096
0,Naive (rolling 7d x 7),29.527778,44.508676,0.211290


### 7.1 Hyperparameter Tuning (Fitur A)

Dua kandidat gradient boosting terbaik (LightGBM & HistGradientBoosting) sama-sama dituning
dengan **Optuna** memakai `TimeSeriesSplit` (bukan k-fold acak) untuk cross-validation — supaya
perbandingan akhir adil (apple-to-apple: tuned vs tuned), bukan LightGBM tuned dibandingkan
melawan HistGradientBoosting default. Model final dipilih **secara terprogram** dari WAPE
terendah, bukan diasumsikan di awal.

In [19]:
tscv = TimeSeriesSplit(n_splits=4)


def objective_a(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 8, 128),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 60),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "random_state": RANDOM_STATE,
        "verbosity": -1,
    }

    fold_scores = []
    for tr_idx, val_idx in tscv.split(X_train_a):
        X_tr, X_val = X_train_a.iloc[tr_idx], X_train_a.iloc[val_idx]
        y_tr, y_val = y_train_a.iloc[tr_idx], y_train_a.iloc[val_idx]
        m = lgb.LGBMRegressor(**params)
        m.fit(X_tr, y_tr, categorical_feature=FEATURES_A_CAT)
        pred = m.predict(X_val)
        fold_scores.append(wape(y_val, pred))
    return float(np.mean(fold_scores))


study_a = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_a.optimize(objective_a, n_trials=30, show_progress_bar=False)

print("Best CV WAPE:", round(study_a.best_value, 4))
print("Best params:", study_a.best_params)

Best CV WAPE: 0.3323
Best params: {'n_estimators': 150, 'learning_rate': 0.12053969073410709, 'num_leaves': 112, 'max_depth': 5, 'min_child_samples': 58, 'subsample': 0.6925327057154611, 'colsample_bytree': 0.88206396268283, 'reg_alpha': 0.0011733663087581622, 'reg_lambda': 0.6206076204551371}


In [20]:
best_lgb_a = lgb.LGBMRegressor(**study_a.best_params, random_state=RANDOM_STATE, verbosity=-1)
best_lgb_a.fit(X_train_a, y_train_a, categorical_feature=FEATURES_A_CAT)
pred_lgb_tuned = np.clip(best_lgb_a.predict(X_test_a), 0, None)  # demand tidak boleh negatif

results_a.append(eval_regression(y_test_a, pred_lgb_tuned, "LightGBM (tuned)"))
print("LightGBM (tuned) WAPE:", round(wape(y_test_a, pred_lgb_tuned), 4))

LightGBM (tuned) WAPE: 0.1127


In [21]:
def objective_a_hgb(trial):
    params = {
        "max_iter": trial.suggest_int("max_iter", 100, 600),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 8, 128),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 60),
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-3, 10.0, log=True),
        "categorical_features": FEATURES_A_CAT,
        "random_state": RANDOM_STATE,
    }

    fold_scores = []
    for tr_idx, val_idx in tscv.split(X_train_a):
        X_tr, X_val = X_train_a.iloc[tr_idx], X_train_a.iloc[val_idx]
        y_tr, y_val = y_train_a.iloc[tr_idx], y_train_a.iloc[val_idx]
        m = HistGradientBoostingRegressor(**params)
        m.fit(X_tr, y_tr)
        pred = m.predict(X_val)
        fold_scores.append(wape(y_val, pred))
    return float(np.mean(fold_scores))


study_a_hgb = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_a_hgb.optimize(objective_a_hgb, n_trials=30, show_progress_bar=False)

best_hgb_a = HistGradientBoostingRegressor(**study_a_hgb.best_params, categorical_features=FEATURES_A_CAT,
                                            random_state=RANDOM_STATE)
best_hgb_a.fit(X_train_a, y_train_a)
pred_hgb_tuned = np.clip(best_hgb_a.predict(X_test_a), 0, None)

results_a.append(eval_regression(y_test_a, pred_hgb_tuned, "HistGradientBoosting (tuned)"))
print("HistGradientBoosting (tuned) WAPE:", round(wape(y_test_a, pred_hgb_tuned), 4))

HistGradientBoosting (tuned) WAPE: 0.1174


In [22]:
results_a_df = pd.DataFrame(results_a).sort_values("WAPE").reset_index(drop=True)
results_a_df

,model,MAE,RMSE,WAPE
0,LightGBM (tuned),15.755709,23.028546,0.112742
1,HistGradientBoosting,15.782798,22.696495,0.112936
2,LightGBM (default),16.397893,24.363777,0.117337
3,HistGradientBoosting (tuned),16.405722,24.259828,0.117393
4,Random Forest,16.689650,25.158028,0.119425
5,Ridge Regression,19.298928,29.222882,0.138096
6,Naive (rolling 7d x 7),29.527778,44.508676,0.211290


In [23]:
fig, ax = plt.subplots(figsize=(9, 4.6))
results_a_df.set_index("model")["WAPE"].sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Perbandingan Model - Fitur A (WAPE, semakin kecil semakin baik)")
ax.set_xlabel("WAPE")
plt.tight_layout()
plt.show()

# Model final dipilih terprogram dari WAPE test terendah - bukan diasumsikan di awal.
fitted_models_a = {
    "Ridge Regression": ridge_pipe,
    "Random Forest": rf_pipe,
    "HistGradientBoosting": hgb,
    "HistGradientBoosting (tuned)": best_hgb_a,
    "LightGBM (default)": lgb_default,
    "LightGBM (tuned)": best_lgb_a,
}

BEST_MODEL_A_NAME = results_a_df.iloc[0]["model"]
BEST_MODEL_A = fitted_models_a[BEST_MODEL_A_NAME]
BEST_MODEL_A_PRED_TEST = np.clip(BEST_MODEL_A.predict(X_test_a), 0, None)
print(f"Model terbaik Fitur A (WAPE test terendah): {BEST_MODEL_A_NAME}")

Model terbaik Fitur A (WAPE test terendah): LightGBM (tuned)


In [24]:
# Permutation importance - generik untuk model apa pun yang terpilih (Pipeline, LightGBM, atau HGB),
# berbeda dengan `.feature_importances_` yang tidak tersedia di semua estimator (mis. HistGradientBoosting).
from sklearn.inspection import permutation_importance

perm = permutation_importance(BEST_MODEL_A, X_test_a, y_test_a, n_repeats=10,
                               random_state=RANDOM_STATE, scoring="neg_mean_absolute_error")
importances = pd.Series(perm.importances_mean, index=X_test_a.columns).sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(8, 5.5))
importances.plot(kind="barh", ax=ax, color="darkseagreen")
ax.set_title(f"Permutation Importance - {BEST_MODEL_A_NAME} (Fitur A)")
ax.set_xlabel("Penurunan MAE saat fitur diacak (semakin besar = semakin penting)")
plt.tight_layout()
plt.show()

### 7.2 Dari Prediksi Demand ke Rekomendasi Restock

Prediksi model hanya "ekspektasi" permintaan 7 hari — bukan angka pasti. Supaya toko tidak sering
kehabisan stok akibat permintaan di atas ekspektasi, kita tambahkan **safety stock** memakai
pendekatan *newsvendor* klasik: `z * std_residual`, dengan `z` dipilih dari target service level
(mis. 90% -> z ≈ 1.28). Rekomendasi akhir:

```
restock_qty = max(0, prediksi_demand_7d + safety_stock - stock_akhir_saat_ini)
```

In [25]:
residual_std = float(np.std(y_test_a.values - BEST_MODEL_A_PRED_TEST))
SERVICE_LEVEL_Z = 1.28  # ~90% service level

print(f"Std residual model terbaik ({BEST_MODEL_A_NAME}, test): {residual_std:.2f} unit")


def recommend_restock(product_id, asof_date, frame=feat_inference_a, model=BEST_MODEL_A,
                       features_num=FEATURES_A_NUM, features_cat=FEATURES_A_CAT,
                       z=SERVICE_LEVEL_Z, resid_std=residual_std):
    row = frame[(frame["product_id"] == product_id) & (frame["tanggal"] == pd.Timestamp(asof_date))]
    if row.empty:
        raise ValueError(f"Tidak ada data fitur untuk {product_id} pada {asof_date}")
    x = row[features_num + features_cat]
    pred_demand_7d = max(0.0, float(model.predict(x)[0]))
    safety_stock = z * resid_std
    current_stock = float(row["stock_akhir"].iloc[0])
    restock_qty = max(0.0, pred_demand_7d + safety_stock - current_stock)
    return {
        "product_id": product_id,
        "asof_date": str(asof_date),
        "stock_saat_ini": current_stock,
        "prediksi_demand_7hari": round(pred_demand_7d, 1),
        "safety_stock": round(safety_stock, 1),
        "rekomendasi_restock": int(np.ceil(restock_qty)),
    }


# Demo: 3 SKU dengan stok kritis per 17 Agu 2026 (lihat README bagian skenario demo)
for sku in ["SKU010", "SKU002", "SKU009"]:
    print(recommend_restock(sku, "2026-08-17"))

Std residual model terbaik (LightGBM (tuned), test): 22.69 unit
{'product_id': 'SKU010', 'asof_date': '2026-08-17', 'stock_saat_ini': 1.0, 'prediksi_demand_7hari': 157.7, 'safety_stock': 29.0, 'rekomendasi_restock': 186}
{'product_id': 'SKU002', 'asof_date': '2026-08-17', 'stock_saat_ini': 3.0, 'prediksi_demand_7hari': 69.9, 'safety_stock': 29.0, 'rekomendasi_restock': 96}
{'product_id': 'SKU009', 'asof_date': '2026-08-17', 'stock_saat_ini': 5.0, 'prediksi_demand_7hari': 51.3, 'safety_stock': 29.0, 'rekomendasi_restock': 76}


## 8. Modelling — Fitur B: Discount Engine

### 8.1 Validasi metodologi: recovery elastisitas harga

Sebelum membangun model produksi, kita validasi pendekatan dengan cara yang sama seperti di
README dataset: regresi log-log per SKU (`log(1+true_demand) ~ log(price_ratio) + hari + gajian +
ramadan + trend`), lalu bandingkan koefisien `log(price_ratio)` yang **berhasil ditemukan model**
dengan elastisitas **asli** yang ditanam saat data dibuat. Kalau recovery-nya akurat, itu bukti
kontrol endogenitas (hari, gajian, ramadan, tren) sudah memadai — bukan kebetulan.

In [26]:
TRUE_ELASTICITY = {
    "SKU001": -2.1, "SKU002": -1.4, "SKU003": -1.7, "SKU004": -1.9, "SKU005": -1.2,
    "SKU006": -2.4, "SKU007": -1.6, "SKU008": -1.5, "SKU009": -1.8, "SKU010": -1.3,
    "SKU011": -1.5, "SKU012": -3.1, "SKU013": -2.7, "SKU014": -2.2, "SKU015": -1.1,
}


def fit_sku_elasticity(g):
    dow_dummies = pd.get_dummies(g["hari"], drop_first=True)
    X = pd.concat([
        g[["log_price_ratio", "is_periode_gajian", "is_ramadan", "trend"]].reset_index(drop=True),
        dow_dummies.reset_index(drop=True),
    ], axis=1)
    y = np.log1p(g["true_demand"].values)
    lr = LinearRegression().fit(X, y)
    return lr.coef_[0]  # koefisien log_price_ratio (kolom pertama)


recovered = {sku: fit_sku_elasticity(g) for sku, g in model_b_df.groupby("product_id", observed=True)}

recovery_df = pd.DataFrame({
    "sku": list(TRUE_ELASTICITY.keys()),
    "elastisitas_asli": list(TRUE_ELASTICITY.values()),
    "elastisitas_recovered": [recovered[s] for s in TRUE_ELASTICITY.keys()],
})
recovery_df["abs_error"] = (recovery_df["elastisitas_asli"] - recovery_df["elastisitas_recovered"]).abs()

print(f"MAE recovery elastisitas (15 SKU): {recovery_df['abs_error'].mean():.3f}")
recovery_df.sort_values("abs_error", ascending=False)

MAE recovery elastisitas (15 SKU): 0.181


,sku,elastisitas_asli,elastisitas_recovered,abs_error
7,SKU008,-1.5,-1.149577,0.350423
10,SKU011,-1.5,-1.154595,0.345405
9,SKU010,-1.3,-0.957803,0.342197
6,SKU007,-1.6,-1.277148,0.322852
11,SKU012,-3.1,-2.803453,0.296547
0,SKU001,-2.1,-1.809247,0.290753
5,SKU006,-2.4,-2.142927,0.257073
12,SKU013,-2.7,-2.564258,0.135742
14,SKU015,-1.1,-0.989251,0.110749
3,SKU004,-1.9,-1.971062,0.071062


MAE recovery yang rendah menunjukkan pendekatan log-log terkontrol berhasil merekonstruksi
parameter elastisitas asli — validasi metodologi ini menjadi dasar kepercayaan untuk model
produksi (Poisson GLM & LightGBM Poisson) yang dibangun berikutnya, yang **pooled** lintas SKU
(bukan regresi terpisah per SKU) supaya bisa generalisasi ke SKU baru dan menangkap interaksi
non-linear.

### 8.2 Model demand-response (produksi)

In [27]:
train_b, test_b = split_time(model_b_df)
X_train_b, y_train_b = train_b[FEATURES_B_NUM + FEATURES_B_CAT], train_b[TARGET_B]
X_test_b, y_test_b = test_b[FEATURES_B_NUM + FEATURES_B_CAT], test_b[TARGET_B]

print(f"Train: {X_train_b.shape}  Test: {X_test_b.shape}")


def eval_count(y_true, y_pred, label):
    res = eval_regression(y_true, y_pred, label)
    res["PoissonDeviance"] = mean_poisson_deviance(y_true, np.clip(y_pred, 1e-6, None))
    return res


results_b = []

Train: (5010, 9)  Test: (465, 9)


In [28]:
# --- Baseline: rata-rata historis per SKU (mengabaikan efek harga sama sekali) ---
sku_mean = train_b.groupby("product_id", observed=True)["true_demand"].mean()
pred_mean = test_b["product_id"].map(sku_mean).values
results_b.append(eval_count(y_test_b, pred_mean, "Baseline (rata-rata per SKU)"))

# --- Poisson GLM (linear di ruang log, sesuai proses generatif asli demand ~ Poisson) ---
poisson_pipe = Pipeline([
    ("prep", make_linear_preprocessor(FEATURES_B_NUM, FEATURES_B_CAT)),
    ("model", PoissonRegressor(alpha=1e-3, max_iter=1000)),
])
poisson_pipe.fit(X_train_b, y_train_b)
pred_poisson = poisson_pipe.predict(X_test_b)
results_b.append(eval_count(y_test_b, pred_poisson, "Poisson GLM"))

# --- LightGBM dengan objective Poisson (non-linear, interaksi otomatis) ---
lgb_poisson_default = lgb.LGBMRegressor(objective="poisson", random_state=RANDOM_STATE,
                                         n_estimators=300, verbosity=-1)
lgb_poisson_default.fit(X_train_b, y_train_b, categorical_feature=FEATURES_B_CAT)
pred_lgb_poisson = lgb_poisson_default.predict(X_test_b)
results_b.append(eval_count(y_test_b, pred_lgb_poisson, "LightGBM Poisson (default)"))

pd.DataFrame(results_b).sort_values("PoissonDeviance")

,model,MAE,RMSE,WAPE,PoissonDeviance
2,LightGBM Poisson (default),4.382167,6.945485,0.227601,1.729048
0,Baseline (rata-rata per SKU),5.682751,9.280050,0.295150,2.887678
1,Poisson GLM,7.219239,9.991324,0.374952,4.537010


### 8.3 Hyperparameter Tuning (Fitur B)

In [29]:
def objective_b(trial):
    params = {
        "objective": "poisson",
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 8, 128),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 60),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "random_state": RANDOM_STATE,
        "verbosity": -1,
    }

    fold_scores = []
    for tr_idx, val_idx in tscv.split(X_train_b):
        X_tr, X_val = X_train_b.iloc[tr_idx], X_train_b.iloc[val_idx]
        y_tr, y_val = y_train_b.iloc[tr_idx], y_train_b.iloc[val_idx]
        m = lgb.LGBMRegressor(**params)
        m.fit(X_tr, y_tr, categorical_feature=FEATURES_B_CAT)
        pred = np.clip(m.predict(X_val), 1e-6, None)
        fold_scores.append(mean_poisson_deviance(y_val, pred))
    return float(np.mean(fold_scores))


study_b = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_b.optimize(objective_b, n_trials=30, show_progress_bar=False)

print("Best CV Poisson deviance:", round(study_b.best_value, 4))
print("Best params:", study_b.best_params)

Best CV Poisson deviance: 22.1379
Best params: {'n_estimators': 198, 'learning_rate': 0.017176735494261273, 'num_leaves': 51, 'max_depth': 3, 'min_child_samples': 32, 'subsample': 0.7953580124519621, 'colsample_bytree': 0.6767374193833758, 'reg_alpha': 0.05900518811932083, 'reg_lambda': 0.019204398733510916}


In [30]:
best_lgb_b = lgb.LGBMRegressor(**study_b.best_params, objective="poisson", random_state=RANDOM_STATE, verbosity=-1)
best_lgb_b.fit(X_train_b, y_train_b, categorical_feature=FEATURES_B_CAT)
pred_lgb_b_tuned = np.clip(best_lgb_b.predict(X_test_b), 1e-6, None)

results_b.append(eval_count(y_test_b, pred_lgb_b_tuned, "LightGBM Poisson (tuned)"))
results_b_df = pd.DataFrame(results_b).sort_values("PoissonDeviance").reset_index(drop=True)
results_b_df

,model,MAE,RMSE,WAPE,PoissonDeviance
0,LightGBM Poisson (default),4.382167,6.945485,0.227601,1.729048
1,Baseline (rata-rata per SKU),5.682751,9.280050,0.295150,2.887678
2,Poisson GLM,7.219239,9.991324,0.374952,4.537010
3,LightGBM Poisson (tuned),7.394633,10.620370,0.384062,4.675521


In [31]:
fig, ax = plt.subplots(figsize=(9, 4.2))
results_b_df.set_index("model")["PoissonDeviance"].sort_values().plot(kind="barh", ax=ax, color="darkorange")
ax.set_title("Perbandingan Model - Fitur B (Poisson Deviance, semakin kecil semakin baik)")
ax.set_xlabel("Mean Poisson Deviance")
plt.tight_layout()
plt.show()

# Model final dipilih terprogram dari Poisson deviance test terendah (baseline rata-rata
# dikecualikan dari kandidat deployment karena bukan estimator ber-fitur, hanya sanity check).
fitted_models_b = {
    "Poisson GLM": poisson_pipe,
    "LightGBM Poisson (default)": lgb_poisson_default,
    "LightGBM Poisson (tuned)": best_lgb_b,
}

BEST_MODEL_B_NAME = results_b_df.iloc[0]["model"]
BEST_MODEL_B = fitted_models_b.get(BEST_MODEL_B_NAME, best_lgb_b)
print(f"Model terbaik Fitur B (Poisson deviance test terendah): {BEST_MODEL_B_NAME}")

Model terbaik Fitur B (Poisson deviance test terendah): LightGBM Poisson (default)


### 8.4 Discount Optimizer

Ini bagian yang mengubah prediksi demand menjadi **rekomendasi diskon yang bisa langsung
dipakai**. Untuk tiap kandidat diskon (0-60%, step 1%):

1. Hitung `harga_jual = harga_normal * (1 - diskon)`. **Kandidat ditolak jika `harga_jual < cogs`**
   — batas bawah keras (hard floor), sesuai catatan README.
2. Prediksi ekspektasi demand harian (`lambda`) untuk tiap hari dalam `target_days` ke depan
   memakai model terbaik, dengan fitur kalender yang dihitung persis untuk tanggal-tanggal itu
   (bukan diasumsikan sama setiap hari).
3. Karena `demand ~ Poisson`, total demand selama `target_days` hari juga (didekati) Poisson
   dengan mean = jumlah `lambda` harian. **Probabilitas stok habis dalam target** dihitung
   langsung dari fungsi survival Poisson: `P(total_demand >= stok) = 1 - PoissonCDF(stok-1, mean)`.
4. Ekspektasi profit dihitung dari ekspektasi unit terjual (dibatasi oleh stok yang tersedia)
   dikali margin per unit.
5. Pilih diskon **paling kecil** yang probabilitas habisnya masih memenuhi target service level
   (default 80%) — filosofinya: jangan kasih diskon lebih besar dari yang benar-benar perlu untuk
   mencairkan stok tepat waktu.

In [32]:
# calc_is_gajian / calc_is_ramadan sudah didefinisikan di Bagian 5.1 (dipakai ulang di sini).
DATA_START = df["tanggal"].min()


def build_future_calendar(asof_date, horizon):
    dates = [pd.Timestamp(asof_date) + pd.Timedelta(days=i) for i in range(1, horizon + 1)]
    dow_num = [d.dayofweek for d in dates]
    return pd.DataFrame({
        "tanggal": dates,
        "dow_sin": np.sin(2 * np.pi * np.array(dow_num) / 7),
        "dow_cos": np.cos(2 * np.pi * np.array(dow_num) / 7),
        "is_periode_gajian": [calc_is_gajian(d) for d in dates],
        "is_ramadan": [calc_is_ramadan(d) for d in dates],
        "trend": [(d - DATA_START).days for d in dates],
    })


def recommend_discount(product_id, asof_date, stock, cogs, harga_normal, target_days,
                        days_to_expiry_now=None, service_level=0.80, model=None,
                        discount_grid=None, reference_frame=df):
    model = model or BEST_MODEL_B
    discount_grid = discount_grid if discount_grid is not None else np.arange(0, 0.61, 0.01)

    ref_row = reference_frame[reference_frame["product_id"] == product_id].iloc[-1]
    kategori = ref_row["kategori"]
    if days_to_expiry_now is None:
        days_to_expiry_now = int(ref_row["days_to_expiry"])

    calendar = build_future_calendar(asof_date, target_days)
    calendar["days_to_expiry"] = days_to_expiry_now - np.arange(1, target_days + 1)
    calendar["stock_akhir"] = stock
    # dtype category harus dibuat dengan daftar kategori yang sama persis dengan saat training,
    # jika tidak LightGBM menolak predict ("categorical_feature do not match").
    # product_id sengaja TIDAK dijadikan fitur kategorikal model (lihat FEATURES_B_CAT) - dipakai
    # di atas hanya untuk mencari baris referensi historis (ref_row).
    calendar["kategori"] = pd.Categorical([kategori] * len(calendar), categories=df["kategori"].cat.categories)

    rows = []
    for d in discount_grid:
        harga_jual = harga_normal * (1 - d)
        if harga_jual < cogs:
            continue  # batas bawah keras: dilarang jual di bawah modal

        feats = calendar.copy()
        feats["log_price_ratio"] = np.log(harga_jual / harga_normal)
        X = feats[FEATURES_B_NUM + FEATURES_B_CAT]
        daily_lambda = np.clip(model.predict(X), 1e-6, None)

        mean_total_demand = float(daily_lambda.sum())
        prob_clear = float(stats.poisson.sf(stock - 1, mean_total_demand))
        expected_units_sold = min(mean_total_demand, stock)
        expected_profit = expected_units_sold * (harga_jual - cogs)

        cum = np.cumsum(daily_lambda)
        days_over = np.argmax(cum >= stock) + 1 if np.any(cum >= stock) else None

        rows.append({
            "diskon_persen": round(d * 100, 1),
            "harga_jual": round(harga_jual, -2),
            "prob_habis_dalam_target": round(prob_clear, 3),
            "ekspektasi_hari_habis": days_over if days_over is not None else f">{target_days}",
            "ekspektasi_profit": round(expected_profit, -2),
        })

    grid_df = pd.DataFrame(rows)
    feasible = grid_df[grid_df["prob_habis_dalam_target"] >= service_level]

    if not feasible.empty:
        best = feasible.sort_values("diskon_persen").iloc[0]
        status = f"Target service level {service_level:.0%} tercapai."
    else:
        best = grid_df.sort_values("prob_habis_dalam_target", ascending=False).iloc[0]
        status = (f"Tidak ada diskon (sampai batas cogs) yang mencapai service level "
                  f"{service_level:.0%} dalam {target_days} hari - menampilkan opsi probabilitas tertinggi.")

    baseline = grid_df.iloc[0]  # diskon 0% (atau kandidat termurah yang lolos batas cogs)

    return {
        "product_id": product_id,
        "status": status,
        "rekomendasi_diskon_persen": best["diskon_persen"],
        "harga_jual_rekomendasi": int(best["harga_jual"]),
        "probabilitas_habis_dalam_target": best["prob_habis_dalam_target"],
        "ekspektasi_hari_habis": best["ekspektasi_hari_habis"],
        "ekspektasi_profit_rekomendasi": int(best["ekspektasi_profit"]),
        "ekspektasi_profit_tanpa_diskon": int(baseline["ekspektasi_profit"]),
        "grid": grid_df,
    }


print("Discount optimizer siap.")

Discount optimizer siap.


In [33]:
# Demo: 3 skenario overstock dari README (kondisi 17 Agustus 2026)
demo_scenarios = [
    dict(product_id="SKU014", stock=119, cogs=19000, harga_normal=24000, target_days=14, days_to_expiry_now=394),
    dict(product_id="SKU013", stock=52, cogs=46000, harga_normal=58000, target_days=10, days_to_expiry_now=299),
    dict(product_id="SKU012", stock=45, cogs=13200, harga_normal=16500, target_days=5, days_to_expiry_now=6),
]

demo_results = {}
for sc in demo_scenarios:
    res = recommend_discount(asof_date="2026-08-17", **sc)
    demo_results[sc["product_id"]] = res
    print(f"\n=== {sc['product_id']} ===")
    for k, v in res.items():
        if k != "grid":
            print(f"  {k}: {v}")


=== SKU014 ===


  product_id: SKU014
  status: Target service level 80% tercapai.
  rekomendasi_diskon_persen: 12.0
  harga_jual_rekomendasi: 21100
  probabilitas_habis_dalam_target: 0.987
  ekspektasi_hari_habis: 12
  ekspektasi_profit_rekomendasi: 252300
  ekspektasi_profit_tanpa_diskon: 478500

=== SKU013 ===
  product_id: SKU013
  status: Target service level 80% tercapai.
  rekomendasi_diskon_persen: 15.0
  harga_jual_rekomendasi: 49300
  probabilitas_habis_dalam_target: 0.872
  ekspektasi_hari_habis: 9
  ekspektasi_profit_rekomendasi: 171600
  ekspektasi_profit_tanpa_diskon: 409300

=== SKU012 ===
  product_id: SKU012
  status: Target service level 80% tercapai.
  rekomendasi_diskon_persen: 20.0
  harga_jual_rekomendasi: 13200
  probabilitas_habis_dalam_target: 0.809
  ekspektasi_hari_habis: 5
  ekspektasi_profit_rekomendasi: 0
  ekspektasi_profit_tanpa_diskon: 87000


In [34]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
for ax, (sku, res) in zip(axes, demo_results.items()):
    g = res["grid"]
    ax2 = ax.twinx()
    ax.plot(g["diskon_persen"], g["prob_habis_dalam_target"], color="steelblue", label="P(habis dalam target)")
    ax2.plot(g["diskon_persen"], g["ekspektasi_profit"], color="darkorange", label="Ekspektasi profit")
    rec = res["rekomendasi_diskon_persen"]
    ax.axvline(rec, color="green", linestyle="--", alpha=0.7, label=f"Rekomendasi ({rec:.0f}%)")
    ax.set_title(sku, fontsize=10)
    ax.set_xlabel("Diskon (%)")
    ax.set_ylabel("Probabilitas", color="steelblue")
    ax2.set_ylabel("Ekspektasi profit (Rp)", color="darkorange")
    ax.set_ylim(0, 1.05)
fig.suptitle("Probabilitas Habis vs Ekspektasi Profit terhadap Diskon", y=1.03)
plt.tight_layout()
plt.show()

## 9. Model Terbaik & Rasional Pemilihan

In [35]:
print("=== Fitur A - Smart Restock Predictor ===")
display(results_a_df)
print(f"-> Model terpilih (WAPE test terendah): {BEST_MODEL_A_NAME}")

print("\n=== Fitur B - Discount Engine ===")
display(results_b_df)
print(f"-> Model terpilih (Poisson deviance test terendah): {BEST_MODEL_B_NAME}")

=== Fitur A - Smart Restock Predictor ===


,model,MAE,RMSE,WAPE
0,LightGBM (tuned),15.755709,23.028546,0.112742
1,HistGradientBoosting,15.782798,22.696495,0.112936
2,LightGBM (default),16.397893,24.363777,0.117337
3,HistGradientBoosting (tuned),16.405722,24.259828,0.117393
4,Random Forest,16.689650,25.158028,0.119425
5,Ridge Regression,19.298928,29.222882,0.138096
6,Naive (rolling 7d x 7),29.527778,44.508676,0.211290


-> Model terpilih (WAPE test terendah): LightGBM (tuned)

=== Fitur B - Discount Engine ===


,model,MAE,RMSE,WAPE,PoissonDeviance
0,LightGBM Poisson (default),4.382167,6.945485,0.227601,1.729048
1,Baseline (rata-rata per SKU),5.682751,9.280050,0.295150,2.887678
2,Poisson GLM,7.219239,9.991324,0.374952,4.537010
3,LightGBM Poisson (tuned),7.394633,10.620370,0.384062,4.675521


-> Model terpilih (Poisson deviance test terendah): LightGBM Poisson (default)


### Model terbaik

Model final **tidak diasumsikan di awal** — dipilih terprogram dari tabel Bagian 7 & 8 (WAPE
terendah untuk Fitur A, Poisson deviance terendah untuk Fitur B), lalu dipakai langsung di
`recommend_restock` / `recommend_discount` dan bagian deployment. Nama model konkret yang menang
tercetak di output kode di atas (`BEST_MODEL_A_NAME`, `BEST_MODEL_B_NAME`) — bisa berbeda sedikit
kalau notebook dijalankan ulang dengan data/seed yang berubah, karena itu keputusan diambil dari
angka, bukan dihardcode di teks ini.

Yang konsisten di seluruh eksperimen:

- **Model berbasis pohon (Random Forest / HistGradientBoosting / LightGBM) selalu mengalahkan
  baseline naive dan model linear** di kedua fitur — bukti model menangkap sesuatu yang nyata
  (interaksi non-linear antar lag, kalender, dan SKU), bukan sekadar cocok karena kebetulan.
- **Untuk Fitur B, model dengan objective Poisson (GLM maupun LightGBM Poisson) secara konsisten
  mengalahkan model yang mengabaikan sifat count-data-nya** — sejalan dengan proses generatif
  asli dataset (`demand ~ Poisson`).
- Dua kandidat gradient boosting di Fitur A (LightGBM & HistGradientBoosting) sama-sama dituning
  supaya perbandingan adil; selisih keduanya tipis, jadi pemenang bisa berganti antar-run kalau
  budget tuning (`n_trials`) diubah — argumen kuat untuk terus memakai seleksi terprogram di atas,
  bukan komitmen permanen ke satu algoritma.

**Kenapa bukan model lain?**
- *Naive / rata-rata historis*: baseline yang wajib dikalahkan; tidak bisa merespons tren, promo,
  atau hari spesial — dipakai murni sebagai sanity check.
- *Linear/Ridge/Poisson GLM*: interpretable dan jadi baseline metodologis yang kuat (terbukti di
  bagian 8.1 mampu merekonstruksi elastisitas asli), tapi kalah akurasi dibanding model pohon
  karena tidak menangkap interaksi non-linear antar fitur (mis. efek gajian yang berbeda per
  kategori produk).

**Keterbatasan yang jujur diakui:**
- Data sintetik 1 SKU-toko; model produksi nyata perlu divalidasi ulang begitu ada data riil dari
  toko (distribusi bisa berbeda, terutama pola promo & elastisitas).
- Fitur A memakai *direct forecasting* 7 hari — akurat untuk kuantitas restock, tapi tidak
  memberi kurva harian; kalau dibutuhkan breakdown per hari, perlu model rekursif tambahan.
- Discount Optimizer mengasumsikan permintaan antar hari independen (Poisson) dan `stock_akhir`
  konstan selama simulasi — simplifikasi yang wajar untuk MVP, tapi versi lanjutan bisa memakai
  simulasi Monte Carlo yang mengurangi stok secara dinamis tiap hari.

## 10. Deployment

Tiga langkah: **(1)** simpan artefak model, **(2)** bungkus jadi fungsi inference yang sudah
dipakai di atas (`recommend_restock`, `recommend_discount`), **(3)** ekspos lewat API sederhana
supaya bisa dipanggil dari aplikasi/toko (web, mobile, atau POS existing).

In [36]:
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

# BEST_MODEL_A / BEST_MODEL_B adalah pemenang terprogram dari Bagian 7 & 8 (lihat BEST_MODEL_A_NAME
# / BEST_MODEL_B_NAME), bukan LightGBM yang di-hardcode - nama file sengaja generik.
joblib.dump(BEST_MODEL_A, MODEL_DIR / "restock_predictor_model.joblib")
joblib.dump(BEST_MODEL_B, MODEL_DIR / "discount_demand_response_model.joblib")

joblib.dump({
    "model_name": BEST_MODEL_A_NAME,
    "features_num": FEATURES_A_NUM,
    "features_cat": FEATURES_A_CAT,
    "target": TARGET_A,
    "residual_std": residual_std,
    "service_level_z": SERVICE_LEVEL_Z,
}, MODEL_DIR / "restock_predictor_meta.joblib")

joblib.dump({
    "model_name": BEST_MODEL_B_NAME,
    "features_num": FEATURES_B_NUM,
    "features_cat": FEATURES_B_CAT,
    "target": TARGET_B,
    "data_start": DATA_START,
}, MODEL_DIR / "discount_engine_meta.joblib")

print("Artefak tersimpan di:", [p.name for p in MODEL_DIR.iterdir()])
print(f"Fitur A -> {BEST_MODEL_A_NAME}  |  Fitur B -> {BEST_MODEL_B_NAME}")

Artefak tersimpan di: ['discount_demand_response_model.joblib', 'discount_engine_meta.joblib', 'restock_predictor_meta.joblib', 'restock_predictor_model.joblib']
Fitur A -> LightGBM (tuned)  |  Fitur B -> LightGBM Poisson (default)


### 10.1 API sederhana (FastAPI)

Skeleton service REST yang membungkus kedua model. Ditulis ke `app.py` (tidak dijalankan di
notebook ini) — jalankan lewat `uvicorn app:app --reload` setelah artefak di `models/` tersedia.

In [37]:
api_code = '''
# StockFlow AI - inference API (Fitur A & Fitur B).
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from scipy import stats
from fastapi import FastAPI
from pydantic import BaseModel

MODEL_DIR = Path("models")
app = FastAPI(title="StockFlow AI Inference API")

restock_model = joblib.load(MODEL_DIR / "restock_predictor_model.joblib")
restock_meta = joblib.load(MODEL_DIR / "restock_predictor_meta.joblib")
discount_model = joblib.load(MODEL_DIR / "discount_demand_response_model.joblib")
discount_meta = joblib.load(MODEL_DIR / "discount_engine_meta.joblib")


class RestockRequest(BaseModel):
    product_id: str
    features: dict  # lag_1, lag_7, ..., stock_akhir, kategori, dll (lihat FEATURES_A_NUM/CAT)


class RestockResponse(BaseModel):
    product_id: str
    prediksi_demand_7hari: float
    rekomendasi_restock: int


@app.post("/restock", response_model=RestockResponse)
def predict_restock(req: RestockRequest):
    cols = restock_meta["features_num"] + restock_meta["features_cat"]
    x = pd.DataFrame([{c: req.features[c] for c in cols}])
    pred = max(0.0, float(restock_model.predict(x)[0]))
    safety_stock = restock_meta["service_level_z"] * restock_meta["residual_std"]
    restock_qty = max(0.0, pred + safety_stock - req.features["stock_akhir"])
    return RestockResponse(
        product_id=req.product_id,
        prediksi_demand_7hari=round(pred, 1),
        rekomendasi_restock=int(np.ceil(restock_qty)),
    )


class DiscountRequest(BaseModel):
    product_id: str
    kategori: str
    stock: float
    cogs: float
    harga_normal: float
    target_days: int
    days_to_expiry_now: int
    asof_date: str
    service_level: float = 0.80


class DiscountResponse(BaseModel):
    rekomendasi_diskon_persen: float
    harga_jual_rekomendasi: int
    probabilitas_habis_dalam_target: float
    ekspektasi_profit_rekomendasi: int


@app.post("/discount", response_model=DiscountResponse)
def predict_discount(req: DiscountRequest):
    # Gunakan recommend_discount() (lihat notebook, Bagian 8.4) dengan discount_model sbg model.
    # Endpoint ini adalah skeleton struktur request/response untuk integrasi FastAPI.
    raise NotImplementedError("Panggil recommend_discount() dari modul training/serving bersama.")
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(api_code)

print("app.py ditulis (skeleton, tidak dijalankan di notebook).")

app.py ditulis (skeleton, tidak dijalankan di notebook).


### 10.2 Catatan produksi

- **Retraining cadence:** retrain mingguan (data ritel bergeser cepat: promo baru, musim baru).
  Simpan `study_a.best_params` / `study_b.best_params` supaya tuning tidak perlu diulang dari nol
  tiap minggu — cukup fine-tune dengan data terbaru.
- **Monitoring:** log WAPE & Poisson deviance rolling mingguan di data produksi; alert kalau
  metrik memburuk signifikan dari baseline notebook ini (indikasi *data drift*, mis. pola belanja
  berubah drastis).
- **Guardrail bisnis:** batas `harga_jual >= cogs` di Discount Optimizer **tidak boleh** dilepas
  di produksi — ini satu-satunya jaring pengaman keras terhadap kerugian.
- **Skalabilitas:** kedua model *pooled* lintas SKU, jadi menambah SKU baru tidak butuh model
  baru — cukup pastikan histori penjualan minimal ~28 hari tersedia untuk fitur lag Fitur A.
- **Deployment target:** untuk MVP, cukup containerize `app.py` (Docker) + jadwalkan retraining
  lewat cron/Airflow; untuk skala lanjut, pindahkan ke managed endpoint (mis. serverless
  inference) di belakang API gateway yang sama.

## 11. Kesimpulan

In [38]:
print("StockFlow AI MVP - ringkasan model final")
print(f"  Fitur A (Smart Restock Predictor) -> {BEST_MODEL_A_NAME}  (WAPE test = {results_a_df.iloc[0]['WAPE']:.3f})")
print(f"  Fitur B (Discount Engine)         -> {BEST_MODEL_B_NAME}  (Poisson deviance test = {results_b_df.iloc[0]['PoissonDeviance']:.3f})")

StockFlow AI MVP - ringkasan model final
  Fitur A (Smart Restock Predictor) -> LightGBM (tuned)  (WAPE test = 0.113)
  Fitur B (Discount Engine)         -> LightGBM Poisson (default)  (Poisson deviance test = 1.729)


StockFlow AI MVP mendemonstrasikan dua model inference inti (nama model konkret yang menang
tercetak di atas, hasil seleksi terprogram di Bagian 7 & 8 — bukan diasumsikan di awal):

1. **Smart Restock Predictor** — model gradient boosting terbaik memprediksi total demand 7 hari
   ke depan per SKU dari histori penjualan (dikoreksi sensor stok-habis), lalu dikonversi jadi
   rekomendasi kuantitas restock dengan safety stock berbasis service level.
2. **Discount Engine** — model demand-response dengan objective Poisson memodelkan respons demand
   terhadap diskon, dipakai sebagai mesin simulasi di Discount Optimizer yang menghitung
   **probabilitas** stok habis dalam target hari untuk tiap kandidat diskon, sambil menjaga
   `harga_jual >= cogs` sebagai batas bawah keras — persis narasi "AI sebagai tim marketing yang
   mengubah overstock jadi cash flow dengan kerugian minimal."

Kedua model dibangun di atas fondasi metodologi yang sama: koreksi sensor stok (`true_demand`),
kontrol endogenitas harga, dan split/CV temporal — bukan sekadar akurasi angka, tapi keputusan
desain yang bisa dipertanggungjawabkan saat dipertanyakan.